In [ ]:
import csv
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from PIL import Image
from ultralytics import YOLO

## Дообучение детектора на вине, ищем этикетку

In [56]:
model = YOLO("yolov8n.pt")

In [ ]:
results = model.train(
    data="data/full_dataset_yolo/data.yaml",
    epochs=50,
    imgsz=512,
    batch=4,
    device=0,
    workers=2,
    patience=10,
    project="runs_detect",
    name="wine_labels_v1",
    pretrained=True,
    verbose=True,
    amp=True,
)

New https://pypi.org/project/ultralytics/8.4.158 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.157  Python-3.11.9 torch-2.14.0+cu132 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=full_dataset_yolo/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, mode

In [58]:
model = YOLO(r"runs/detect/runs_detect/wine_labels_v1-11/weights/best.pt")
metrics = model.val(data="full_dataset_yolo/data.yaml")
print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)

Ultralytics 8.4.157  Python-3.11.9 torch-2.14.0+cu132 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 959.6970.5 MB/s, size: 342.3 KB)
val: Scanning C:\Users\Sergey\Documents\VS Code Projects\lct-2026\full_dataset_yolo\val\labels.cache... 39 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 39/39  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 1.6s/it 4.7s0.7s5s
                   all         39         60          1      0.844      0.921        0.8
Speed: 1.2ms preprocess, 5.6ms inference, 0.0ms loss, 3.0ms postprocess per image
Results saved to C:\Users\Sergey\Documents\VS Code Projects\lct-2026\runs\detect\val-7
mAP50: 0.9211908394843871
mAP50-95: 0.8004211785192658


## Сохранение фото с детекцией

In [ ]:
paths = sorted(Path("catalog_images").glob("*.webp"))
out_dir = Path("ref_vis")
out_dir.mkdir(exist_ok=True)

log = []
problems = []

for i, p in enumerate(paths):
    img = Image.open(p).convert("RGB")
    r = model(img, conf=0.25, verbose=False)[0]

    vis = np.array(img).copy()
    n = len(r.boxes)
    confs = []
    best_conf = 0.0
    best_box = None

    for b in r.boxes:
        x0, y0, x1, y1 = b.xyxy[0].cpu().numpy().astype(int)
        c = float(b.conf[0])
        confs.append(c)
        if c > best_conf:
            best_conf = c
            best_box = (int(x0), int(y0), int(x1), int(y1))
        cv2.rectangle(vis, (x0, y0), (x1, y1), (0, 255, 0), 4)
        cv2.putText(
            vis, f"{c:.2f}", (x0, y0 - 8), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2
        )

    # Сохраняем визуализацию
    Image.fromarray(vis).save(out_dir / f"{p.stem}.jpg", quality=80)

    log.append(
        {
            "slug": p.stem,
            "path": str(p),
            "n_boxes": n,
            "best_conf": round(best_conf, 3),
            "best_box": best_box,
            "all_confs": [round(c, 3) for c in confs],
        }
    )

    if n == 0 or n > 1 or best_conf < 0.5:
        problems.append(p.stem)

    if (i + 1) % 100 == 0:
        print(f"{i + 1}/{len(paths)}")

with open("ref_detections.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(
        f, fieldnames=["slug", "path", "n_boxes", "best_conf", "best_box", "all_confs"]
    )
    w.writeheader()
    w.writerows(log)


with open("ref_problems.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(problems))

print(f"готово. всего: {len(log)}, проблем: {len(problems)}")

100/2090
200/2090
300/2090
400/2090
500/2090
600/2090
700/2090
800/2090
900/2090
1000/2090
1100/2090
1200/2090
1300/2090
1400/2090
1500/2090
1600/2090
1700/2090
1800/2090
1900/2090
2000/2090
готово. всего: 2090, проблем: 10


In [60]:
df = pd.read_csv("ref_detections.csv")

print("всего:", len(df))
print()
print("n_boxes == 0:", (df["n_boxes"] == 0).sum())
print("n_boxes == 1:", (df["n_boxes"] == 1).sum())
print("n_boxes > 1 :", (df["n_boxes"] > 1).sum())
print()
print("n_boxes == 1 и conf < 0.5:", ((df["n_boxes"] == 1) & (df["best_conf"] < 0.5)).sum())
print("n_boxes == 1 и conf >= 0.5:", ((df["n_boxes"] == 1) & (df["best_conf"] >= 0.5)).sum())
print()

print("распределение n_boxes:")
print(df["n_boxes"].value_counts().sort_index())
print()

print("распределение best_conf (только где n_boxes >= 1):")
print(df[df["n_boxes"] >= 1]["best_conf"].describe())

всего: 2090

n_boxes == 0: 0
n_boxes == 1: 2080
n_boxes > 1 : 10

n_boxes == 1 и conf < 0.5: 0
n_boxes == 1 и conf >= 0.5: 2080

распределение n_boxes:
n_boxes
1    2080
2       9
3       1
Name: count, dtype: int64

распределение best_conf (только где n_boxes >= 1):
count    2090.000000
mean        0.948652
std         0.024120
min         0.515000
25%         0.941000
50%         0.951000
75%         0.961000
max         0.990000
Name: best_conf, dtype: float64
